In [1]:
import pandas as pd
import numpy as np


df = pd.read_csv("/content/Telcom-Customer-Churn.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Column names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecur

In [3]:
# Remove accidental spaces from column names
df.columns = df.columns.str.strip()

# Find columns without depending on capitalization
column_lookup = {
    col.lower(): col for col in df.columns
}

required = [
    "tenure",
    "monthlycharges",
    "totalcharges",
    "churn"
]

missing = [
    col for col in required
    if col not in column_lookup
]

if missing:
    raise ValueError(
        f"Required columns not found: {missing}\n"
        f"Available columns: {df.columns.tolist()}"
    )

# Select the actual column names from the dataset
tenure_col = column_lookup["tenure"]
monthly_col = column_lookup["monthlycharges"]
total_col = column_lookup["totalcharges"]
target_col = column_lookup["churn"]

# Prepare the three features
X = df[
    [tenure_col, monthly_col, total_col]
].copy()

X.columns = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

X["TotalCharges"] = pd.to_numeric(
    X["TotalCharges"], errors="coerce"
)

X["tenure"] = pd.to_numeric(
    X["tenure"], errors="coerce"
)

X["MonthlyCharges"] = pd.to_numeric(
    X["MonthlyCharges"], errors="coerce"
)

# Prepare the target
y = df[target_col].astype(str).str.strip().str.lower()

y = y.map({
    "no": 0,
    "yes": 1,
    "0": 0,
    "1": 1
})

# Remove rows with missing or invalid values
valid = X.notna().all(axis=1) & y.notna()

X = X.loc[valid]
y = y.loc[valid].astype(int)

print("Features prepared:", X.columns.tolist())
print("Number of usable records:", len(X))

print("\nTarget distribution:")
print(y.value_counts())

Features prepared: ['tenure', 'MonthlyCharges', 'TotalCharges']
Number of usable records: 7032

Target distribution:
Churn
0    5163
1    1869
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

Training records: 5625
Testing records: 1407


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [6]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.7747

Confusion Matrix:
[[927 106]
 [211 163]]

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.90      0.85      1033
           1       0.61      0.44      0.51       374

    accuracy                           0.77      1407
   macro avg       0.71      0.67      0.68      1407
weighted avg       0.76      0.77      0.76      1407



In [7]:
import pickle

with open("churn_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved as churn_model.pkl")

Model saved as churn_model.pkl


In [8]:
with open("churn_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

sample = X_test.iloc[[0]]

prediction = loaded_model.predict(sample)[0]
probability = loaded_model.predict_proba(sample)[0][1]

print("Sample input:")
display(sample)

print("Predicted class:", prediction)
print("Estimated churn probability:", f"{probability:.2%}")

Sample input:


,tenure,MonthlyCharges,TotalCharges
974,59,75.95,4542.35


Predicted class: 0
Estimated churn probability: 7.07%


In [9]:
# !pip install -q pyngrok streamlit

# from getpass import getpass
# from pyngrok import ngrok
# import subprocess
# import time
# import requests

# # Enter your private ngrok token
# token = getpass("Enter your ngrok auth token: ")
# ngrok.set_auth_token(token)

# # Start Streamlit
# process = subprocess.Popen([
#     "streamlit", "run", "app.py",
#     "--server.port", "8501",
#     "--server.address", "0.0.0.0",
#     "--server.headless", "true"
# ])

# # Wait for Streamlit to start
# for attempt in range(20):
#     try:
#         response = requests.get(
#             "http://127.0.0.1:8501/_stcore/health",
#             timeout=2
#         )
#         if response.status_code == 200:
#             break
#     except requests.RequestException:
#         pass
#     time.sleep(1)
# else:
#     raise RuntimeError(
#         "Streamlit did not start. Check the app.py code and logs."
#     )

# # Create a public tunnel
# tunnel = ngrok.connect(8501, "http")
# print("Open your Streamlit application here:")
# print(tunnel.public_url)

In [10]:
import os

print(os.listdir("/content"))

['.config', 'Telcom-Customer-Churn.csv', 'churn_model.pkl', 'sample_data']


In [11]:
%pip install -q streamlit pyngrok

In [12]:
from pyngrok import ngrok
from getpass import getpass

NGROK_TOKEN = getpass("Enter your ngrok auth token: ")
ngrok.set_auth_token(NGROK_TOKEN)

print("ngrok authentication configured.")

Enter your ngrok auth token: ··········
ngrok authentication configured.


In [13]:
import subprocess

log_file = open("/content/logs.txt", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

print("Streamlit is starting...")

Streamlit is starting...


In [14]:
import time
import requests

for attempt in range(20):
    try:
        response = requests.get(
            "http://127.0.0.1:8501/_stcore/health",
            timeout=2
        )

        if response.status_code == 200:
            print("Streamlit is ready!")
            break

    except requests.RequestException:
        time.sleep(1)
else:
    print(open("/content/logs.txt").read())

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: /content/app.py



In [15]:
public_url = ngrok.connect(8501)
print("Open your Churn Prediction App:")
print(public_url)

Open your Churn Prediction App:
NgrokTunnel: "https://overdress-matron-move.ngrok-free.dev" -> "http://localhost:8501"


In [16]:
import streamlit as st
import pickle
import pandas as pd

# Load the trained model
with open("churn_model.pkl", "rb") as f:
    model = pickle.load(f)

st.title("Telecom Customer Churn Prediction")

st.write("Enter customer details to predict churn.")

tenure = st.slider("Tenure (months)", 0, 72, 12)
monthly_charges = st.number_input(
    "Monthly Charges", min_value=0.0, value=70.0
)
total_charges = st.number_input(
    "Total Charges", min_value=0.0, value=840.0
)

if st.button("Predict Churn"):
    features = pd.DataFrame(
        [[tenure, monthly_charges, total_charges]],
        columns=["tenure", "MonthlyCharges", "TotalCharges"]
    )

    prediction = model.predict(features)[0]

    if prediction == 1:
        st.error("The customer is likely to churn.")
    else:
        st.success("The customer is likely to stay.")

2026-10-30 09:36:02.588 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-30 09:36:03.078 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-10-30 09:36:03.082 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-30 09:36:03.083 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-30 09:36:03.085 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-30 09:36:03.085 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-30 09:36:03.086 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-30 09:36:03.087 Thread 'MainThread': mi